# 3. Prediction — iteration 3 updated
Loads the exact completed fit from notebook 2 with integrity checks. CUDA inference uses float32
for stable distribution decoding. Exports daily means and optional individual Q70 values in blocks;
model/segment/month totals use **PRED_MEAN only**. No rounding before aggregation.
No business-target multipliers or historical-share overrides are applied.
For a backtest run, actuals are read only after predictions and scored on identical IDs and dates.


In [ ]:
from pathlib import Path
import json, hashlib, uuid, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()

def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def check_daily(frame, date_col, start, end, label):
    idx = pd.DatetimeIndex(pd.to_datetime(frame[date_col]))
    expected = pd.date_range(start, end, freq='D')
    if idx.has_duplicates or not idx.equals(expected):
        raise ValueError(f'{label}: duplicate, missing or unexpected dates; '
                         f'expected {len(expected)} daily rows, got {len(idx)}. '
                         'Repair the source grid; missing observations are not assumed to be zero.')

SEGMENTS = {
    'SPLENDOR+':'100 CC', 'HF DELUXE':'100 CC', 'HF 100':'100 CC', 'PASSION':'100 CC',
    'GLAMOUR':'125 CC', 'SUPER SPLENDOR':'125 CC', 'XTREME 125':'125 CC',
    'XPULSE':'PREMIUM', 'XTREME 160':'PREMIUM', 'XTREME 250':'PREMIUM',
    'DESTINI':'SCOOTER', 'PLEASURE+':'SCOOTER', 'XOOM':'SCOOTER',
}


In [ ]:
PROJECT_DIR = Path.cwd()  # same folder as the chunking notebook
RUN_DIR_OVERRIDE = None  # optional absolute path to a completed data snapshot
RUN_DIR = Path(RUN_DIR_OVERRIDE or read_json(PROJECT_DIR/'iteration3_active_run.json')['run_dir'])
if not (RUN_DIR/'DATA_READY').exists():
    raise RuntimeError('Data snapshot is incomplete')
cfg = read_json(RUN_DIR/'config.json')
dm = read_json(RUN_DIR/'data_manifest.json')
for name, expected in [('config.json',dm['config_hash']),('calendar.parquet',dm['calendar_hash']),
                       ('selected_series.parquet',dm['membership_hash'])]:
    if digest(RUN_DIR/name) != expected:
        raise RuntimeError(f'Data snapshot changed: {name}; create a new run')
TIME, KEY, TARGET = cfg['time_col'], cfg['group_col'], cfg['target_col']
STATIC, FUTURE = cfg['static_covariates'], cfg['future_covariates']
ICL, OCL = cfg['icl'], cfg['ocl']
TRAIN_START, TRAIN_END = pd.Timestamp(cfg['train_start']), pd.Timestamp(cfg['train_end'])
HISTORY_END = pd.Timestamp(cfg['val_end'])
FC_START, FC_END = pd.Timestamp(cfg['forecast_start']), pd.Timestamp(cfg['forecast_end'])
VAL_OUTPUT_START = pd.Timestamp(cfg['val_output_start'])
VAL_INPUT_START = VAL_OUTPUT_START - pd.Timedelta(days=ICL)
assert HISTORY_END + pd.Timedelta(days=1) == FC_START
assert OCL == (FC_END-FC_START).days+1
assert VAL_OUTPUT_START > TRAIN_END
print('Data snapshot:', RUN_DIR, '| Population:', cfg['population'])


In [ ]:
import torch, darts, inspect, pickle, gc
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
import pytorch_lightning as pl
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not visible. Select your CUDA-enabled PyTorch environment/kernel. '
                       'This notebook will not silently fall back to CPU.')
torch.set_float32_matmul_precision('high')
PRECISION = 'bf16-mixed' if torch.cuda.is_bf16_supported() else '32-true'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('Darts:', darts.__version__, '| PyTorch:', torch.__version__, '| precision:', PRECISION)


In [ ]:
FIT_DIR_OVERRIDE = None  # optional explicit fit directory belonging to RUN_DIR
active = read_json(RUN_DIR/'active_fit.json')
FIT_DIR = Path(FIT_DIR_OVERRIDE or active['fit_dir'])
if FIT_DIR_OVERRIDE is None and digest(FIT_DIR/'bundle.json')!=active['bundle_hash']:
    raise RuntimeError('Bundle metadata changed')
bundle = read_json(FIT_DIR/'bundle.json')
if bundle['run_id']!=cfg['run_id'] or bundle['config_hash']!=dm['config_hash'] or bundle['calendar_hash']!=dm['calendar_hash']:
    raise RuntimeError('Checkpoint and data configuration do not match')
if darts.__version__!=bundle['versions']['darts']:
    raise RuntimeError('Use the same Darts version as training: '+bundle['versions']['darts'])
for name,expected in bundle['file_hashes'].items():
    if digest(FIT_DIR/name)!=expected: raise RuntimeError(f'Artifact changed: {name}')
CACHE_DIR = FIT_DIR/'series_cache'
records = read_json(FIT_DIR/'cache_manifest.json')
for r in records:
    if digest(CACHE_DIR/r['file'])!=r['sha256']: raise RuntimeError('Cached target changed: '+r['key'])
with open(FIT_DIR/'static_encoded.pkl','rb') as f: encoded=pickle.load(f)
static_raw = pd.read_parquet(FIT_DIR/'static_raw.parquet')
if len(encoded)!=len(records) or static_raw[KEY].tolist()!=[r['key'] for r in records]:
    raise RuntimeError('Series/static order mismatch')
SHARED_COV = TimeSeries.from_pickle(FIT_DIR/'shared_cov.pkl')
if list(SHARED_COV.components)!=FUTURE or SHARED_COV.end_time()<FC_END:
    raise RuntimeError('Future covariate schema or horizon mismatch')
MODEL_NAME=bundle['model_name']
model=TFTModel.load_from_checkpoint(model_name=MODEL_NAME,work_dir=bundle['work_dir'],
                                   best=True,map_location='cpu')
if model.input_chunk_length!=ICL or model.output_chunk_length!=OCL:
    raise RuntimeError('Checkpoint horizon mismatch')
if not isinstance(model.likelihood,NegativeBinomialLikelihood):
    raise TypeError('Expected NegativeBinomialLikelihood')
if sum(p.numel() for p in model.model.input_embeddings.parameters())<=0:
    raise RuntimeError('Expected categorical embeddings')
BATCH_SIZE=64
BLOCK_SIZE=1000
EXPORT_INDIVIDUAL_Q70=True
trainer=pl.Trainer(accelerator='gpu',devices=1,precision='32-true',logger=False,
                   enable_checkpointing=False,enable_progress_bar=False)
OUT_DIR=FIT_DIR/('predictions_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_')+uuid.uuid4().hex[:8])
OUT_DIR.mkdir(exist_ok=False)
print('Checkpoint:',MODEL_NAME,'| Dates:',FC_START.date(),'through',FC_END.date())


In [ ]:
from collections.abc import Sequence
from functools import lru_cache

class Targets(Sequence):
    def __init__(self, records, static_frames, split):
        self.records, self.static_frames, self.split = records, static_frames, split
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        if isinstance(i,slice): return [self[j] for j in range(*i.indices(len(self)))]
        if i<0: i += len(self)
        if not 0<=i<len(self): raise IndexError(i)
        return self._get(i)
    @lru_cache(maxsize=128)
    def _get(self,i):
        with np.load(CACHE_DIR/self.records[i]['file'], allow_pickle=False) as z:
            dates = pd.DatetimeIndex(z['dates'])
            sales = z['sales']
        if self.split == 'train': mask = dates <= TRAIN_END
        elif self.split == 'val': mask = (dates>=VAL_INPUT_START)&(dates<=HISTORY_END)
        elif self.split == 'history': mask = (dates>=FC_START-pd.Timedelta(days=ICL))&(dates<=HISTORY_END)
        else: raise ValueError(self.split)
        return TimeSeries.from_times_and_values(dates[mask],sales[mask,None],columns=[TARGET],
                                                static_covariates=self.static_frames[i])

class Shared(Sequence):
    def __init__(self, ts, n): self.ts,self.n=ts,n
    def __len__(self): return self.n
    def __getitem__(self,i):
        if isinstance(i,slice): return [self.ts for _ in range(*i.indices(self.n))]
        if i<0: i+=self.n
        if not 0<=i<self.n: raise IndexError(i)
        return self.ts


In [ ]:
from scipy import stats

def summarise(p_ts):
    expected=[f'{TARGET}_r',f'{TARGET}_p']
    if list(p_ts.components)!=expected: raise ValueError(f'Expected {expected}, got {list(p_ts.components)}')
    params=p_ts.values(copy=False).astype(np.float64)
    r,prob=params[:,0],params[:,1]
    if not np.isfinite(params).all() or (r<=0).any() or ((prob<=0)|(prob>=1)).any():
        raise ValueError('Invalid distribution parameters; do not silently clip them')
    ans={'PRED_MEAN':r*prob/(1-prob)}
    if EXPORT_INDIVIDUAL_Q70: ans['INDIVIDUAL_Q70']=stats.nbinom.ppf(.70,r,1-prob)
    if not all(np.isfinite(v).all() for v in ans.values()): raise ValueError('Invalid decoded predictions')
    return ans

# Private methods are used ONLY to verify the installed Darts parameter convention.
# Stop on incompatible versions rather than interpreting a parameter name as the mean.
raw=torch.tensor([[[[-2.,-1.]],[[.4,-.2]],[[3.,1.5]]]],dtype=torch.float64)
lk=model.likelihood
dist=lk._distr_from_params(lk._params_from_output(raw))
exported=lk.predict_likelihood_parameters(raw).detach().cpu().numpy().reshape(3,2)
probe=TimeSeries.from_times_and_values(pd.date_range('2000-01-01',periods=3),exported,
                                      columns=[f'{TARGET}_r',f'{TARGET}_p'])
np.testing.assert_allclose(summarise(probe)['PRED_MEAN'],dist.mean.numpy().ravel(),rtol=1e-10,atol=1e-10)
counts=torch.tensor([0.,2.,5.],dtype=torch.float64).reshape(1,3,1)
np.testing.assert_allclose(stats.nbinom.logpmf(counts.numpy().ravel(),exported[:,0],1-exported[:,1]),
                           dist.log_prob(counts).numpy().ravel(),rtol=1e-10,atol=1e-10)
print('Verified analytical mean and probability convention against installed likelihood')


In [ ]:
metadata=static_raw[[KEY,'MODEL_FAMILY']].copy()
metadata['SEGMENT']=metadata.MODEL_FAMILY.map(cfg['segments'])
if metadata.SEGMENT.isna().any(): raise ValueError('Missing segment mapping')
if metadata[KEY].duplicated().any(): raise ValueError('Duplicate metadata IDs')
model_sums,segment_sums,day_sums,files=[],[],[],[]
row_count=0
for lo in range(0,len(records),BLOCK_SIZE):
    hi=min(lo+BLOCK_SIZE,len(records))
    history=Targets(records[lo:hi],encoded[lo:hi],'history')
    for ts in history:
        if len(ts)!=ICL or ts.end_time()!=FC_START-pd.Timedelta(days=1):
            raise ValueError('History not aligned to forecast start')
    predictions=model.predict(n=OCL,series=history,future_covariates=Shared(SHARED_COV,hi-lo),
        predict_likelihood_parameters=True,num_samples=1,batch_size=BATCH_SIZE,trainer=trainer,
        dataloader_kwargs={'num_workers':0,'pin_memory':True},verbose=False)
    if len(predictions)!=hi-lo: raise ValueError('Missing series predictions')
    rows=[]
    for record,p in zip(records[lo:hi],predictions):
        if len(p)!=OCL or p.start_time()!=FC_START or p.end_time()!=FC_END:
            raise ValueError('Unexpected forecast dates')
        rows.append(pd.DataFrame({KEY:record['key'],TIME:p.time_index,**summarise(p)}))
    frame=pd.concat(rows,ignore_index=True).merge(metadata,on=KEY,how='left',validate='many_to_one')
    if frame.duplicated([KEY,TIME]).any() or len(frame)!=(hi-lo)*OCL:
        raise ValueError('Incorrect prediction coverage')
    frame['MONTH']=frame[TIME].dt.to_period('M').astype(str)
    path=OUT_DIR/f'daily_block_{lo//BLOCK_SIZE:04d}.parquet'
    frame.to_parquet(path,index=False)
    files.append({'path':path.name,'rows':len(frame),'sha256':digest(path)})
    model_sums.append(frame.groupby(['MODEL_FAMILY','MONTH'],as_index=False).PRED_MEAN.sum())
    segment_sums.append(frame.groupby(['SEGMENT','MONTH'],as_index=False).PRED_MEAN.sum())
    day_sums.append(frame.groupby([TIME,'SEGMENT'],as_index=False).PRED_MEAN.sum())
    row_count+=len(frame)
    print(f'Forecasted {hi:,}/{len(records):,} series')
    history._get.cache_clear()
    del history,predictions,rows,frame
    gc.collect()

if trainer.strategy.root_device.type!='cuda': raise RuntimeError('Prediction did not use CUDA')
if row_count!=len(records)*OCL: raise RuntimeError('Incomplete forecast')
model_month=pd.concat(model_sums).groupby(['MODEL_FAMILY','MONTH'],as_index=False).PRED_MEAN.sum()
segment_month=pd.concat(segment_sums).groupby(['SEGMENT','MONTH'],as_index=False).PRED_MEAN.sum()
segment_day=pd.concat(day_sums).groupby([TIME,'SEGMENT'],as_index=False).PRED_MEAN.sum()
np.testing.assert_allclose(model_month.PRED_MEAN.sum(),segment_month.PRED_MEAN.sum(),rtol=1e-12)
model_month.to_csv(OUT_DIR/'forecast_by_model_mean.csv',index=False)
segment_month.to_csv(OUT_DIR/'forecast_by_segment_mean.csv',index=False)
segment_day.to_parquet(OUT_DIR/'forecast_by_segment_daily_mean.parquet',index=False)
write_json(OUT_DIR/'prediction_manifest.json',dict(model_name=MODEL_NAME,run_id=cfg['run_id'],
    population=cfg['population'],forecast_start=str(FC_START.date()),forecast_end=str(FC_END.date()),
    series_count=len(records),rows=row_count,blocks=files,statistic='sum of analytical means',
    total_mean=float(model_month.PRED_MEAN.sum())))
print(segment_month.groupby('SEGMENT').PRED_MEAN.sum())
print('Total expected sales (lakh):',model_month.PRED_MEAN.sum()/100000)
print('Population:',cfg['population'],'| Days:',OCL,'| Output:',OUT_DIR)
print('Individual Q70 values are diagnostic only, not quantiles of total demand.')


In [ ]:
# Score unseen festive dates only in backtest mode. No forecast adjustment is made.
if cfg['mode']=='festive_backtest_2025':
    actual_parts=[]
    for entry in dm['chunks']:
        if digest(RUN_DIR/entry['path'])!=entry['sha256']: raise RuntimeError('Source chunk changed')
        a=pd.read_parquet(RUN_DIR/entry['path'],columns=[KEY,TIME,TARGET,'MODEL_FAMILY'])
        a[TIME]=pd.to_datetime(a[TIME])
        a=a[a[TIME].between(FC_START,FC_END)]
        actual_parts.append(a)
    actual=pd.concat(actual_parts,ignore_index=True)
    if actual.duplicated([KEY,TIME]).any() or len(actual)!=len(records)*OCL:
        raise ValueError('Actuals do not cover exactly the forecasted series and dates')
    actual=actual.set_index([KEY,TIME])[[TARGET]]
    parts=[]
    for file in files:
        f=pd.read_parquet(OUT_DIR/file['path'])
        j=f.join(actual,on=[KEY,TIME],validate='one_to_one')
        if j[TARGET].isna().any(): raise ValueError('Missing actuals')
        j['ABS_ERROR']=(j.PRED_MEAN-j[TARGET]).abs()
        parts.append(j.groupby('SEGMENT')[[TARGET,'PRED_MEAN','ABS_ERROR']].sum())
    scores=pd.concat(parts).groupby(level=0).sum()
    scores['SERIES_DAY_WAPE']=scores.ABS_ERROR/scores[TARGET].replace(0,np.nan)
    scores['SIGNED_BIAS']=(scores.PRED_MEAN-scores[TARGET])/scores[TARGET].replace(0,np.nan)
    scores.to_csv(OUT_DIR/'festive_backtest_segment_metrics.csv')
    print(scores)
    # Aggregate-level WAPE is separately labelled; aggregation can cancel errors.
    actual_daily_parts=[]
    for a in actual_parts:
        a['SEGMENT']=a.MODEL_FAMILY.map(cfg['segments'])
        actual_daily_parts.append(a.groupby([TIME,'SEGMENT'],as_index=False)[TARGET].sum())
    ad=pd.concat(actual_daily_parts).groupby([TIME,'SEGMENT'],as_index=False)[TARGET].sum()
    comparison=segment_day.merge(ad,on=[TIME,'SEGMENT'],validate='one_to_one')
    comparison['ABS_ERROR']=(comparison.PRED_MEAN-comparison[TARGET]).abs()
    daily_scores=comparison.groupby('SEGMENT')[[TARGET,'PRED_MEAN','ABS_ERROR']].sum()
    daily_scores['SEGMENT_DAY_WAPE']=daily_scores.ABS_ERROR/daily_scores[TARGET].replace(0,np.nan)
    daily_scores.to_csv(OUT_DIR/'festive_backtest_segment_daily_metrics.csv')
else:
    print('Production: future accuracy cannot be measured yet. Run the 2025 backtest preset separately.')
